In [2]:
%cd nanoVLM

/teamspace/studios/this_studio/nanoVLM-action/nanoVLM


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
# sft

!python train_action_sft.py \
  --jsonl /teamspace/studios/this_studio/nanoVLM-action/nanoVLM/data/emptyenv_sft_dataset/dataset.jsonl \
  --images_root /teamspace/studios/this_studio/nanoVLM-action/nanoVLM/data/emptyenv_sft_dataset \
  --batch_size 16 \
  --max_steps 1000 \
  --val_ratio 0.10 \
  --lr_mp 0.005 \
  --lr_head 0.005 \
  --max_length 2048 \
  --log_every 3 \
  --eval_every 30 \
  --out checkpoints_emptyenv_action/base_sft 

Resize to max side len: True
/teamspace/studios/this_studio/nanoVLM-action/nanoVLM/data/emptyenv_sft_dataset/test_unseen_sizes/dataset.jsonl /teamspace/studios/this_studio/nanoVLM-action/nanoVLM/data/emptyenv_sft_dataset/test_unseen_sizes/
Resize to max side len: True
Loading from backbone weights
Successfully loaded google/siglip2-base-patch16-512 weights from safetensors. Model has 86,433,024 parameters.
Extending token embeddings from torch.Size([49152, 960]) to torch.Size([49218, 960])
Initialized 66 new token embeddings
Successfully loaded HuggingFaceTB/SmolLM2-360M-Instruct weights from safetensors. Model has 361,884,480 parameters.
compiling model...
Dataset length: 20000
Fetching sample 0 ...
Fetched sample 0 in 0.014752626419067383 sec
input_ids: torch.Size([107]) label: 0
num image chunks: 1
eval...
eval...
val_loss 0.8775 | val_acc 0.683 
unseen_sizes_loss 0.7353 | unseen_sizes_acc 0.755 
  3%|█                                        | 29/1125 [00:13<04:23,  4.15it/s]step   

In [ ]:
# grpo action unseen 8 9

!python grpo_action_train.py \
   --init_ckpt /teamspace/studios/this_studio/nanoVLM-action/nanoVLM/checkpoints_emptyenv_action/base_sft/step_30 \
   --out checkpoints_emptyenv_action/grpo \
   --sizes 8 9 \
   --eval_sizes 10 \
   --rollout_episodes 32 \
   --ppo_epochs 1 \
   --minibatch_episodes 4 \
   --max_steps_per_ep 100 \
   --temperature 1.4 \
   --eval_every 2 \
   --lr 1e-4 \
   --train_iters 21 

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/teamspace/studios/this_studio/nanoVLM-action/nanoVLM/grpo_action_train.py:272: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
Device: cuda
Resize to max side len: True
 56%|████████████████████████▏                  | 18/32 [00:49<00:34,  2.45s/it]

In [9]:
# grpo lora reasoning + action

!python grpo_train_lora.py \
  --init_ckpt /teamspace/studios/this_studio/nanoVLM-action/nanoVLM/checkpoints_emptyenv_action/base_sft/step_0 \
  --out checkpoints_emptyenv_action/grpo_lora \
  --sizes 8 9 \
  --train_iters 30 \
  --rollout_episodes 32 \
  --ppo_epochs 1 \
  --minibatch_episodes 4 \
  --max_steps_per_ep 100 \
  --temperature 1.4 \
  --lr 1e-4 \
  --reasoning_top_p 0.75 \
  --reasoning_top_k 20 \
  --max_reasoning_tokens 32 \
  --reasoning_temperature 0.5 \
  --reasoning_repetition_penalty 1.2 

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
Device: cuda
LoRA injected: LLM=64 linear(s), MP=1 linear(s)
Trainable params: 928,067 / 461,042,051 (0.2013%)
Resize to max side len: True
/teamspace/studios/this_studio/nanoVLM-action/nanoVLM/grpo_train_lora.py:642: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
  6%|██▊                                         | 2/32 [03:01<45:16, 90.56s/it]
Traceback (most recent call last):
  File "/teamspace/studios/this_studio/nanoVLM-action/nanoVLM/grpo_train_lora.py", line 721, 